# Mixture of Experts from scratch
Top-2 router, MLP experts, load-balancing loss, and a comparison with dense MLPs of equal active params.

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
from dense import make_task
from moe import init_moe, moe_forward, aux_loss, active_params, total_params
from train import train_dense, train_moe

## 1. Multi-cluster task: each cluster has its own labelling rule

In [ ]:
X, y, c = make_task(4000, 1); Xt, yt, ct = make_task(2000, 2)
print(X.shape, 'classes', np.bincount(y), 'clusters', np.bincount(c))

## 2. One forward pass through the MoE layer

In [ ]:
p = init_moe(16, 16, 4, 8, np.random.default_rng(0))
out, cache = moe_forward(p, X[:5], k=2)
print('chosen experts per token:\n', cache['top'])
print('gates (non-zero only for the chosen 2):\n', cache['g'].round(3))
print('active params', active_params(p, 2), 'total params', total_params(p))

## 3. Train: dense (equal active params) vs MoE without and with the aux loss

In [ ]:
_, hd = train_dense(X, y, Xt, yt, 39, 40, 3e-3, 64, 42)
print('dense H=39 acc', hd[-1])
for a in (0.0, 0.1):
    pm, hm = train_moe(X, y, Xt, yt, 16, 8, 2, a, 40, 3e-3, 64, 42)
    _, cache = moe_forward(pm, Xt, 2)
    _, _, f, _ = aux_loss(cache, 8, 2)
    print(f'MoE aux={a}: acc={hm["test_acc"][-1]:.4f}  utilization={f.round(3)}')

## 4. Full sweep
`python run_smoke.py` writes `results/`.

In [ ]:
import json
print(json.dumps(json.load(open('../results/metrics.json'))['headline'], indent=1))